# 🏢 Room Occupancy Estimation Using Environmental Sensors
### Uçtan Uca Makine Öğrenmesi, Keşifçi Veri Analizi ve Özgün Feature Set Deneyleri

---

## 📌 1. Proje Amacı ve Araştırma Sorusu
Akıllı bina otomasyonu, enerji verimliliği ve HVAC (ısıtma, havalandırma, klima) optimizasyonunda bir odadaki kişi sayısının bilinmesi kritik öneme sahiptir. Kameralar gibi invaziv (mahremiyeti ihlal eden) yöntemler yerine **çevresel sensörler** (Sıcaklık, Işık, Ses, $\text{CO}_2$, $\text{CO}_2$ Değişim Eğimi ve PIR Hareket Sensörleri) kullanılarak oda doluluğu tespit edilebilir mi?

### 🎯 Ana Araştırma Sorusu:
> **"Çevresel sensörlerden elde edilen veriler kullanılarak bir odadaki kişi sayısı ($0, 1, 2, 3$ kişi) ne kadar doğru tahmin edilebilir? Daha az sensör veya özetlenmiş özniteliklerle benzer performansa ulaşılabilir mi?"**

### 🔬 Projenin Metodolojik Öncelikleri:
1. **Zaman Serisi Duyarlılığı (Zero Temporal Leakage):** Sensör verileri 30 saniyelik ardışık ölçümlerden oluştuğu için rastgele değil, **kronolojik (temporal) train/test split** (%80 Train, %20 Test) kullanılacaktır.
2. **Sağlam Preprocessing Pipeline'ları:** `StandardScaler` yalnızca eğitim verisine fit edilecek, test verisine yalnızca transform uygulanacaktır.
3. **Çok Sınıflı Değerlendirme:** Sınıf dengesizliği nedeniyle sadece *Accuracy* değil, *Macro Precision, Macro Recall, Macro F1, Weighted F1* ve *Confusion Matrix* incelenecektir.
4. **Feature Set Ablation:** Sensörlerin fiziksel katkısı (PIR'sız, Yalnızca Çevresel, Özetlenmiş, Zaman Bilgili) sistematik olarak karşılaştırılacaktır.


## 🛠️ 2. Kütüphaneler ve Ortam Kurulumu

In [1]:
import sys
from pathlib import Path

# Proje ana dizinini Python path'ine ekle
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from src import config
from src.data_loader import load_raw_data, inspect_data_quality, print_quality_report, temporal_train_test_split, stratified_random_split
from src.feature_engineering import build_full_feature_dataset, get_feature_sets
from src.models import get_model_zoo, get_decision_tree_pipeline
from src.visualization import (
    plot_class_distribution,
    plot_sensor_distributions,
    plot_sensor_vs_occupancy_boxplots,
    plot_correlation_heatmap,
    plot_temporal_trend,
    plot_confusion_matrices,
    plot_feature_importance,
    plot_feature_set_ablation,
    plot_temporal_vs_random_split
)

# Grafik görsel ayarları
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("✅ Kütüphaneler ve modüller başarıyla yüklendi!")


✅ Kütüphaneler ve modüller başarıyla yüklendi!


## 🔍 3. Veri Setinin Yüklenmesi ve Veri Kalitesi Kontrolleri
Veri seti UCI Machine Learning Repository üzerinden alınan **Room Occupancy Estimation** veri setidir.
İlk olarak ham veri setinin boyutları, eksik değerler, duplicate kayıtlar ve veri tipleri kontrol edilmektedir.


In [2]:
# Ham veriyi yükle
df_raw = load_raw_data()
print(f"Dataset Boyutu: {df_raw.shape[0]:,} satır, {df_raw.shape[1]} sütun")
display(df_raw.head())


Dataset Boyutu: 10,129 satır, 19 sütun


,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,2017/12/22,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,2017/12/22,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,2017/12/22,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,2017/12/22,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,2017/12/22,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1


In [3]:
# Veri kalitesi ve yapısal kontrolleri çalıştır
quality_report = inspect_data_quality(df_raw)
print_quality_report(quality_report)


DATA QUALITY & INTEGRITY REPORT
Total Rows: 10,129
Total Columns: 19
Date Range: 2017-12-22 10:49:41 to 2018-01-11 09:00:09
Median Sampling Interval: 31.0 seconds
Missing Values (NaN): 0
Infinite Values: 0
Exact Duplicate Rows: 0
------------------------------------------------------------
Target Class Distribution (Room_Occupancy_Count):
  Class 0 (0 persons): 8,228 records (81.23%)
  Class 1 (1 persons): 459 records (4.53%)
  Class 2 (2 persons): 748 records (7.38%)
  Class 3 (3 persons): 694 records (6.85%)
------------------------------------------------------------
Physical Non-Negative Checks:
  All physical sensor columns satisfy non-negative bounds.


In [4]:
# Nümerik değişkenlerin betimsel istatistikleri
df_raw.describe().round(3).T


,count,mean,std,min,25%,50%,75%,max
S1_Temp,10129.0,25.454,0.351,24.940,25.190,25.38,25.63,26.380
S2_Temp,10129.0,25.546,0.586,24.750,25.190,25.38,25.63,29.000
S3_Temp,10129.0,25.057,0.427,24.440,24.690,24.94,25.38,26.190
S4_Temp,10129.0,25.754,0.356,24.940,25.440,25.75,26.00,26.560
S1_Light,10129.0,25.445,51.011,0.000,0.000,0.00,12.00,165.000
S2_Light,10129.0,26.016,67.304,0.000,0.000,0.00,14.00,258.000
S3_Light,10129.0,34.248,58.401,0.000,0.000,0.00,50.00,280.000
S4_Light,10129.0,13.220,19.602,0.000,0.000,0.00,22.00,74.000
S1_Sound,10129.0,0.168,0.317,0.060,0.070,0.08,0.08,3.880
S2_Sound,10129.0,0.120,0.267,0.040,0.050,0.05,0.06,3.440


## 📊 4. Keşifçi Veri Analizi (Exploratory Data Analysis - EDA)
Bu bölümde hedef değişkenin dağılımı, sensörlerin dağılım özellikleri, kişi sayısına göre sensör tepkileri ve zaman serisi dinamikleri incelenmektedir.


### 4.1 Hedef Sınıf Dağılımı (`Room_Occupancy_Count`)
Veri setinde 0, 1, 2 ve 3 kişi sınıfları bulunmaktadır. Sınıf dengesizliğini görselleştirelim:

In [5]:
plot_class_distribution(df_raw)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\01_class_distribution.png


### 4.2 Sensör Dağılımları (Histogram & KDE)
Sıcaklık, Işık, Ses, $\text{CO}_2$, $\text{CO}_2$ Slope ve PIR hareket sensörlerinin tekil dağılımları:

In [6]:
plot_sensor_distributions(df_raw)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\02_sensor_distributions.png


### 4.3 Kişi Sayısına Göre Sensör Değişimi (Boxplot)
Odada kişi sayısı arttıkça sensör değerlerinde meydana gelen değişim:

In [7]:
plot_sensor_vs_occupancy_boxplots(df_raw)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\03_sensor_vs_occupancy_boxplots.png


### 4.4 Korelasyon Matrisi (Correlation Heatmap)
Tüm nümerik sensörler ile oda doluluğu arasındaki korelasyon ilişkileri:

In [8]:
plot_correlation_heatmap(df_raw)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\04_correlation_heatmap.png


### 4.5 Zaman Serisi Trendi (Temporal Dynamics)
Örnek zaman aralığında Kişi Sayısı, Işık ve $\text{CO}_2$ seviyelerinin zaman eksenindeki değişimi:

In [9]:
plot_temporal_trend(df_raw)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\05_temporal_trend.png


## ⚙️ 5. Öznitelik Mühendisliği (Feature Engineering)
Sensörlerin fiziksel konumlarından ve zaman yapısından yararlanarak şu değişkenleri üretiyoruz:
1. **Mekânsal Ortalama (Spatial Mean):** `Avg_Temp`, `Avg_Light`, `Avg_Sound`
2. **Mekânsal Fark (Spatial Range):** `Temp_Range`, `Light_Range`, `Sound_Range` (Maksimum - Minimum)
3. **Zaman Değişkenleri (Temporal Context):** `Hour`, `Minute`, `DayOfWeek`


In [10]:
# Feature engineering uygula
df_processed = build_full_feature_dataset(df_raw)
print(f"İşlenmiş Veri Seti Boyutu: {df_processed.shape[0]:,} satır, {df_processed.shape[1]} sütun")
display(df_processed[["Avg_Temp", "Temp_Range", "Avg_Light", "Light_Range", "Avg_Sound", "Sound_Range", "Hour", "Minute", "DayOfWeek"]].head())


İşlenmiş Veri Seti Boyutu: 10,129 satır, 28 sütun


,Avg_Temp,Temp_Range,Avg_Light,Light_Range,Avg_Sound,Sound_Range,Hour,Minute,DayOfWeek
0,24.9075,0.82,62.00,87,0.0975,0.13,10,49,4
1,24.9225,0.88,61.75,88,0.2750,0.88,10,50,4
2,24.9225,0.94,62.00,87,0.1700,0.37,10,50,4
3,24.9375,0.88,62.00,87,0.1750,0.32,10,51,4
4,24.9375,0.88,62.25,87,0.0900,0.12,10,51,4


In [11]:
# Tanımlanan Feature Setleri
feature_sets = get_feature_sets()
for name, cols in feature_sets.items():
    print(f"• {name} ({len(cols)} features): {cols}")


• Feature Set A (All Sensors) (16 features): ['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR']
• Feature Set B (No PIR) (14 features): ['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope']
• Feature Set C (Environmental Only) (14 features): ['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound', 'S5_CO2', 'S5_CO2_Slope']
• Feature Set D (Engineered Summary) (10 features): ['Avg_Temp', 'Temp_Range', 'Avg_Light', 'Light_Range', 'Avg_Sound', 'Sound_Range', 'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR']
• Feature Set E (Sensors + Time) (19 features): ['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light', 'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Soun

## ⏱️ 6. Metodolojik Karar: Kronolojik (Temporal) Train / Test Split
Veriler ~31 saniyede bir toplandığı için rastgele split gelecekteki anlık bilgiyi geçmişe sızdırır (**Temporal Data Leakage**).
Bu nedenle:
* İlk %80 kronolojik gözlem $\rightarrow$ **Eğitim Seti (Train)** ($8,103$ satır)
* Son %20 kronolojik gözlem $\rightarrow$ **Test Seti (Test)** ($2,026$ satır)


In [12]:
train_df, test_df = temporal_train_test_split(df_processed, save_to_disk=False)
print(f"Eğitim Seti: {len(train_df):,} satır | Test Seti: {len(test_df):,} satır")

print("\nEğitim Seti Sınıf Dağılımı (%):")
print((train_df[config.TARGET_COL].value_counts(normalize=True)*100).round(2).to_dict())

print("\nTest Seti Sınıf Dağılımı (%):")
print((test_df[config.TARGET_COL].value_counts(normalize=True)*100).round(2).to_dict())


Eğitim Seti: 8,103 satır | Test Seti: 2,026 satır

Eğitim Seti Sınıf Dağılımı (%):
{0: 79.93, 2: 8.08, 3: 6.32, 1: 5.66}

Test Seti Sınıf Dağılımı (%):
{0: 86.43, 3: 8.98, 2: 4.59}


## 🤖 7. Deney 1: Temel Makine Öğrenmesi Algoritmaları Karşılaştırması
Tüm sensörleri içeren **Feature Set A (16 feature)** üzerinde üç temel algoritma test edilmektedir:
1. **Logistic Regression** (StandardScaler Pipeline)
2. **K-Nearest Neighbors** ($k \in [3, 5, 7]$, StandardScaler Pipeline)
3. **Decision Tree** ($\text{depth} \in [3, 5, 7, 10]$)


In [13]:
from src.evaluate import evaluate_model_pipeline

features_a = config.FEATURE_SET_A
X_train_a, y_train = train_df[features_a], train_df[config.TARGET_COL]
X_test_a, y_test = test_df[features_a], test_df[config.TARGET_COL]

model_zoo = get_model_zoo()
benchmark_results = []
cm_dict = {}
fitted_models = {}

for name, pipeline in model_zoo.items():
    res = evaluate_model_pipeline(pipeline, X_train_a, y_train, X_test_a, y_test)
    benchmark_results.append({
        "Model": name,
        "Train Acc": res["train_accuracy"],
        "Test Acc": res["accuracy"],
        "Macro Prec": res["macro_precision"],
        "Macro Recall": res["macro_recall"],
        "Macro F1": res["macro_f1"],
        "Weighted F1": res["weighted_f1"]
    })
    cm_dict[name] = res["confusion_matrix"]
    fitted_models[name] = res["fitted_pipeline"]

benchmark_df = pd.DataFrame(benchmark_results).sort_values("Macro F1", ascending=False).reset_index(drop=True)
display(benchmark_df.style.format({
    "Train Acc": "{:.2%}",
    "Test Acc": "{:.2%}",
    "Macro Prec": "{:.2%}",
    "Macro Recall": "{:.2%}",
    "Macro F1": "{:.2%}",
    "Weighted F1": "{:.2%}"
}).highlight_max(subset=["Macro F1", "Test Acc"], color="#d4edda"))


,Model,Train Acc,Test Acc,Macro Prec,Macro Recall,Macro F1,Weighted F1
0,Decision Tree (depth=5),99.56%,95.06%,88.42%,73.19%,76.85%,94.66%
1,Decision Tree (depth=7),99.84%,95.06%,88.42%,73.19%,76.85%,94.66%
2,Decision Tree (depth=10),99.96%,95.06%,88.42%,73.19%,76.85%,94.66%
3,Decision Tree (depth=3),96.88%,92.05%,83.24%,61.04%,67.85%,90.94%
4,Logistic Regression,99.62%,95.26%,88.59%,66.81%,61.58%,93.63%
5,KNN (k=3),99.65%,93.48%,55.90%,46.34%,46.85%,93.08%
6,KNN (k=5),99.53%,92.65%,51.49%,43.61%,44.18%,92.44%
7,KNN (k=7),99.36%,91.41%,44.77%,40.43%,41.83%,91.52%


## 🔬 8. Deney 2: Feature Set Ablation Çalışması
Farklı sensör kombinasyonlarının ve özetlenmiş özniteliklerin model başarısına etkisini incelemek için Decision Tree ($	ext{depth}=5$) modelini sabit tutarak 5 farklı Feature Set'i karşılaştırıyoruz:
* **Set A (All Sensors - 16 Değişken)**: Tüm sensörler
* **Set B (No PIR - 14 Değişken)**: Hareket sensörleri hariç
* **Set C (Environmental Only - 14 Değişken)**: Pasif çevresel sensörler
* **Set D (Engineered Summary - 10 Değişken)**: Ortalama ve range özetleri
* **Set E (Sensors + Time - 19 Değişken)**: Sensörler + Saat/Gün bilgisi


In [14]:
ablation_results = []
for set_name, cols in feature_sets.items():
    pipe = get_decision_tree_pipeline(max_depth=5)
    res = evaluate_model_pipeline(pipe, train_df[cols], y_train, test_df[cols], y_test)
    ablation_results.append({
        "Feature Set": set_name,
        "Feature Count": len(cols),
        "Accuracy": res["accuracy"],
        "Macro Precision": res["macro_precision"],
        "Macro Recall": res["macro_recall"],
        "Macro F1": res["macro_f1"],
        "Weighted F1": res["weighted_f1"]
    })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df.style.format({
    "Accuracy": "{:.2%}",
    "Macro Precision": "{:.2%}",
    "Macro Recall": "{:.2%}",
    "Macro F1": "{:.2%}",
    "Weighted F1": "{:.2%}"
}))


,Feature Set,Feature Count,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Feature Set A (All Sensors),16,95.06%,88.42%,73.19%,76.85%,94.66%
1,Feature Set B (No PIR),14,92.05%,83.24%,61.04%,67.85%,90.94%
2,Feature Set C (Environmental Only),14,92.05%,83.24%,61.04%,67.85%,90.94%
3,Feature Set D (Engineered Summary),10,91.61%,49.79%,55.52%,52.34%,89.91%
4,Feature Set E (Sensors + Time),19,95.95%,89.42%,76.98%,78.98%,95.62%


In [15]:
plot_feature_set_ablation(ablation_df)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\08_feature_set_ablation.png


## ⚠️ 9. Deney 3: Veri Sızıntısı Analizi (Temporal Split vs. Random Split)
Zaman serisi verilerinde rastgele split yapıldığında birbirini 30 saniye arayla takip eden benzer satırlar hem eğitim hem test setine dağılır ve yapay olarak yüksek skorlar elde edilir. Bu durumu sayısal olarak kanıtlayalım:

In [16]:
# 1. Temporal split skoru
pipe_t = get_decision_tree_pipeline(max_depth=5)
res_t = evaluate_model_pipeline(pipe_t, train_df[features_a], y_train, test_df[features_a], y_test)

# 2. Stratified Random split skoru
train_r, test_r = stratified_random_split(df_processed)
pipe_r = get_decision_tree_pipeline(max_depth=5)
res_r = evaluate_model_pipeline(pipe_r, train_r[features_a], train_r[config.TARGET_COL], test_r[features_a], test_r[config.TARGET_COL])

split_comparison_df = pd.DataFrame([
    {"Split Strategy": "Chronological (Temporal 80/20)", "Accuracy": res_t["accuracy"], "Macro F1": res_t["macro_f1"], "Weighted F1": res_t["weighted_f1"]},
    {"Split Strategy": "Stratified Random (Random 80/20)", "Accuracy": res_r["accuracy"], "Macro F1": res_r["macro_f1"], "Weighted F1": res_r["weighted_f1"]}
])

display(split_comparison_df.style.format({"Accuracy": "{:.2%}", "Macro F1": "{:.2%}", "Weighted F1": "{:.2%}"}))


,Split Strategy,Accuracy,Macro F1,Weighted F1
0,Chronological (Temporal 80/20),95.06%,76.85%,94.66%
1,Stratified Random (Random 80/20),99.56%,98.37%,99.56%


In [17]:
plot_temporal_vs_random_split(split_comparison_df)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\09_temporal_vs_random_split.png


## 🎯 10. Hata Analizi ve Karışıklık Matrisleri (Confusion Matrices)
Modellerin hangi sınıfları birbirine karıştırdığını (özellikle 2 kişi ile 3 kişi ayrımı) inceleyelim:

In [18]:
selected_cms = {
    "Logistic Regression": cm_dict["Logistic Regression"],
    "KNN (k=5)": cm_dict["KNN (k=5)"],
    "Decision Tree (depth=5)": cm_dict["Decision Tree (depth=5)"],
    "Decision Tree (depth=7)": cm_dict["Decision Tree (depth=7)"]
}
plot_confusion_matrices(selected_cms)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\06_confusion_matrices.png


## 💡 11. Öznitelik Önem Düzeyleri (Feature Importance)
Decision Tree ($	ext{depth}=5$) modelinin oda doluluğunu tahmin ederken hangi değişkenlerden daha fazla yararlandığını sıralayalım:

In [19]:
best_dt = fitted_models["Decision Tree (depth=5)"].named_steps["classifier"]
plot_feature_importance(config.FEATURE_SET_A, best_dt.feature_importances_)
plt.show()


Saved: C:\Users\gokde\Desktop\bootcamp\project_medium\outputs\figures\07_feature_importance.png


## 📝 12. Sonuçlar, Tartışma ve Limitasyonlar

### 📌 Temel Bulgular:
1. **En Başarılı Algoritma:** Sınıf dengesizliği altında en yüksek genellenebilirliği **Decision Tree ($\text{depth}=5$)** göstermiştir (Test Accuracy: **%95.06**, Macro F1: **%76.85**).
2. **PIR Sensörlerinin Rolü:** PIR hareket sensörleri çıkarıldığında (Set B) Macro F1 skoru **%76.85**'ten **%67.85**'e gerilemiştir. Bu, pasif çevresel ölçümlerin hareket anlarını yakalamada PIR desteğine ihtiyaç duyduğunu gösterir.
3. **CO₂ ve Işık Sensörlerinin Önemi:** Feature importance analizinde $\text{CO}_2$ Değişim Eğimi (`S5_CO2_Slope`) ve Işık (`S1_Light`) modeller için en belirleyici öznitelikler olmuştur.
4. **Zaman Serisi Split Farkı:** Rastgele split yapıldığında yapay olarak **%99.56** gibi gerçekçi olmayan bir accuracy elde edilirken, zamansal split ile gerçek dünya başarısının **%95.06** olduğu doğrulanmıştır.

### ⚠️ Limitasyonlar:
* Veri tek bir kontrollü oda ortamından toplanmıştır; farklı oda büyüklükleri ve yalıtım koşullarında modellerin yeniden kalibre edilmesi gerekir.
* Test setinde 1 kişilik gözlemlerin bulunmaması zaman serisi veri setlerinin doğal dinamiklerinden kaynaklanmaktadır.
* Kişi sayısı 0–3 aralığı ile sınırlıdır.
